In [1]:
import math
import random

CITIES = {
    "Delhi": (28.61, 77.21), "Jaipur": (26.91, 75.79), "Agra": (27.18, 78.01),
    "Lucknow": (26.85, 80.95), "Patna": (25.59, 85.14), "Kolkata": (22.57, 88.36),
    "Bhopal": (23.26, 77.41), "Nagpur": (21.15, 79.09), "Ahmedabad": (23.02, 72.57),
    "Mumbai": (19.08, 72.88), "Pune": (18.52, 73.86), "Hyderabad": (17.39, 78.49),
    "Bengaluru": (12.97, 77.59), "Chennai": (13.08, 80.27), "Kochi": (9.93, 76.27),
}
NAMES = list(CITIES)
N = len(NAMES)

def km(i, j):
    lat1, lon1 = map(math.radians, CITIES[NAMES[i]])
    lat2, lon2 = map(math.radians, CITIES[NAMES[j]])
    a = (math.sin((lat2 - lat1) / 2) ** 2
         + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2)
    return 6371 * 2 * math.asin(math.sqrt(a))

DIST = [[km(i, j) for j in range(N)] for i in range(N)]   

def tour_length(tour):
    return sum(DIST[tour[i]][tour[(i + 1) % N]] for i in range(N))

def random_tour():
    tour = list(range(N))
    random.shuffle(tour)
    return tour

def neighbour(tour, i, j):
    return tour[:i] + tour[i:j + 1][::-1] + tour[j + 1:]

def hill_climbing(tour):
    current, cur_len = tour, tour_length(tour)
    while True:
        best, best_len = None, cur_len
        for i in range(N - 1):                   
            for j in range(i + 1, N):
                cand = neighbour(current, i, j)
                cand_len = tour_length(cand)
                if cand_len < best_len:
                    best, best_len = cand, cand_len
        if best is None:                          
            return current, cur_len               
        current, cur_len = best, best_len

def random_restart_hill_climbing(restarts=10):
    results = [hill_climbing(random_tour()) for _ in range(restarts)]
    return min(results, key=lambda r: r[1])

def simulated_annealing(tour, T=300.0, cooling=0.9995, T_min=0.1):
    current, cur_len = tour, tour_length(tour)
    best, best_len = current, cur_len
    while T > T_min:
        i, j = sorted(random.sample(range(N), 2))    
        cand = neighbour(current, i, j)
        cand_len = tour_length(cand)
        dE = cur_len - cand_len                      
        if dE > 0 or random.random() < math.exp(dE / T):
            current, cur_len = cand, cand_len        
            if cur_len < best_len:
                best, best_len = current, cur_len
        T *= cooling                                  
    return best, best_len

def show(tour):
    return " -> ".join(NAMES[c] for c in tour + [tour[0]])

if __name__ == "__main__":
    random.seed(1)

    print("=" * 74)
    print("WORKED EXAMPLE: Sunil's delivery tour through 15 cities")
    print("=" * 74)
    start = random_tour()
    print(f"Random starting tour : {tour_length(start):7.1f} km")
    print(f"                       {show(start)}\n")

    hc_tour, hc_len = hill_climbing(start)
    print(f"Hill Climbing        : {hc_len:7.1f} km")
    print(f"                       {show(hc_tour)}\n")

    sa_tour, sa_len = simulated_annealing(start)
    print(f"Simulated Annealing  : {sa_len:7.1f} km")
    print(f"                       {show(sa_tour)}")

    print("\n" + "=" * 74)
    print("ANALYSIS: 30 runs of each method from different random starts")
    print("=" * 74)
    runs = 30
    hc = [hill_climbing(random_tour())[1] for _ in range(runs)]
    rr = [random_restart_hill_climbing(10)[1] for _ in range(runs)]
    sa = [simulated_annealing(random_tour())[1] for _ in range(runs)]

    best_known = min(hc + rr + sa)

    def row(name, lens):
        hit = sum(1 for x in lens if abs(x - best_known) < 1e-6)
        print(f"{name:<26} | {min(lens):8.1f} | {sum(lens)/len(lens):8.1f} | "
              f"{max(lens):8.1f} | {hit:2d}/{runs}")

    print(f"{'Method':<26} | {'Best km':>8} | {'Average':>8} | {'Worst':>8} | Reached best known")
    print("-" * 74)
    row("Hill Climbing", hc)
    row("Hill Climbing + restarts", rr)
    row("Simulated Annealing", sa)
    print(f"{'Best known tour':<26} | {best_known:8.1f}")

    print("\nObservations:")
    print(" - Hill Climbing is very fast but can get stuck in a local optimum, so its worst and average results are worse.") 
    print(" - Random restarts fix this, but they cost about 10 times more work (10 hill-climbing runs for every answer).")
    print(" - Simulated Annealing accepts some worse moves early on (high T), so with ONE run it beats a single hill climb: better average and much better worst case.")

WORKED EXAMPLE: Sunil's delivery tour through 15 cities
Random starting tour : 16632.7 km
                       Kochi -> Pune -> Delhi -> Chennai -> Bhopal -> Kolkata -> Lucknow -> Ahmedabad -> Nagpur -> Hyderabad -> Patna -> Jaipur -> Bengaluru -> Mumbai -> Agra -> Kochi

Hill Climbing        :  6951.6 km
                       Delhi -> Jaipur -> Bhopal -> Nagpur -> Ahmedabad -> Mumbai -> Pune -> Kochi -> Bengaluru -> Chennai -> Hyderabad -> Kolkata -> Patna -> Lucknow -> Agra -> Delhi

Simulated Annealing  :  6717.6 km
                       Jaipur -> Delhi -> Agra -> Lucknow -> Patna -> Kolkata -> Bhopal -> Nagpur -> Hyderabad -> Chennai -> Bengaluru -> Kochi -> Pune -> Mumbai -> Ahmedabad -> Jaipur

ANALYSIS: 30 runs of each method from different random starts
Method                     |  Best km |  Average |    Worst | Reached best known
--------------------------------------------------------------------------
Hill Climbing              |   6664.0 |   6773.9 |   7088.9 | 11/30
